<a href="https://colab.research.google.com/github/23f1000190/Important-colab-or-notebooks/blob/main/Music_Genre_classification_project_iter2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install wandb -q

In [3]:
!pip install librosa

In [4]:
# import data from kaggle
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"priyanshu23f1000190","key":"4cb91d988321b70f3a30e18c67778a30"}'}

In [5]:
!mkdir -p /root/.config/kaggle
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.config/kaggle/kaggle.json': No such file or directory


In [8]:
ls -l /root/.config/kaggle/

total 0


In [7]:
os.environ["KAGGLE_USERNAME"] = "priyanshu23f1000190"
os.environ["KAGGLE_KEY"] = "4cb91d988321b70f3a30e18c67778a30"

In [9]:

path = kagglehub.competition_download("jan-2026-dl-gen-ai-project")
print("Dataset downloaded to:", path)

100%|██████████| 16.1G/16.1G [11:22<00:00, 25.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project


In [6]:
# @title
# Required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import librosa
import wandb
import random
import kagglehub

from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# @title
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @title
#Initialze W&B
wandb.login(relogin=True)
wandb.init(
    project= "DL-23f100190-notebook-t12026",
    entity= "23f1000190-dl-genai-project",
    name="EffNetB0_full__pipeline4",
    config= {
        "model": "efficientnet_b0",
        "lr": 1e-4,
        "batch_size": 16,
        "epochs":12,
        "segment_seconds": 10,
        "mel_bins":128
    }

)

config =wandb.config

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
#build Dataset Index
data_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
os.listdir(data_path)

['ESC-50-master',
 'genres_stems',
 'mashups',
 'test.csv',
 'sample_submission.csv']

In [ ]:
base_path= os.path.join(data_path, "genres_stems")
genres= sorted(os.listdir(base_path))
genres

['blues',
 'classical',
 'country',
 'disco',
 'hiphop',
 'jazz',
 'metal',
 'pop',
 'reggae',
 'rock']

In [ ]:
genre_to_label={g : i for i, g in enumerate(genres)}
genre_to_label

{'blues': 0,
 'classical': 1,
 'country': 2,
 'disco': 3,
 'hiphop': 4,
 'jazz': 5,
 'metal': 6,
 'pop': 7,
 'reggae': 8,
 'rock': 9}

In [ ]:
label_to_genre = {v: k for k , v in genre_to_label.items()}

In [ ]:
samples =[]
for g in genres:
    genre_path = os.path.join(base_path,g)
    for folder in os.listdir(genre_path):
        samples.append((os.path.join(genre_path, folder), genre_to_label[g]))# e.g append genre> discso> all undrlying foelre 1, 2 and so on


train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    stratify=[s[1] for s in samples],
    random_state=42
)

In [ ]:
#SpecAugment - elper function

def spec_augment(mel, time_mask=30, freq_mask=15):
    mel = mel.copy()
    # time masking
    t = mel.shape[1]
    t0= random.randint(0, max(1, t-time_mask))
    mel[:, t0: t0 + time_mask]=0
    #frequency masking
    f = mel.shape[0]
    f0=random.randint(0, max(1, f - freq_mask))
    mel[f0:f0 + freq_mask, :] = 0

    return mel


In [ ]:
# Dataset class (segment + stem mix + Augment + mix up)
class GenreDataset(Dataset):

    def __init__(self, samples, genre_to_label, segment_seconds=10, augment=False):
        self.samples = samples
        self.genre_to_label = genre_to_label
        self.segment_len = segment_seconds * 44100
        self.augment = augment

        # Map genre to folders for cross-stem mixing
        self.genre_to_folders = {}
        for folder, label in samples:
            self.genre_to_folders.setdefault(label, []).append(folder)

    def load_stem(self, folder_path, stem):
        path = os.path.join(folder_path, f"{stem}.wav")
        waveform, _ = librosa.load(path, sr=44100)
        return waveform.astype(np.float32)

    def random_segment(self, waveform):
        if len(waveform) <= self.segment_len:
            return np.pad(waveform, (0, self.segment_len - len(waveform)))
        start = random.randint(0, len(waveform) - self.segment_len)
        return waveform[start:start+self.segment_len]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        folder_path, label = self.samples[idx]

        # Stem recombination
        if self.augment and random.random() < 0.5:
            folders = self.genre_to_folders[label]
            bass = self.load_stem(random.choice(folders), "bass")
            drums = self.load_stem(random.choice(folders), "drums")
            vocals = self.load_stem(random.choice(folders), "vocals")
            other = self.load_stem(random.choice(folders), "other")
        else:
            bass = self.load_stem(folder_path, "bass")
            drums = self.load_stem(folder_path, "drums")
            vocals = self.load_stem(folder_path, "vocals")
            other = self.load_stem(folder_path, "other")
        min_len = min(len(bass), len(drums), len(vocals), len(other))

        bass = bass[:min_len]
        drums = drums[:min_len]
        vocals = vocals[:min_len]
        other = other[:min_len]
        waveform = np.clip(bass + drums + vocals + other, -1, 1)

        waveform = self.random_segment(waveform)

        # Amplitude scaling
        if self.augment and random.random() < 0.5:
            waveform *= random.uniform(0.7, 1.3)

        mel = librosa.feature.melspectrogram(
            y=waveform,
            sr=44100,
            n_mels=128,
            hop_length=512
        )

        log_mel = librosa.power_to_db(mel)

        # SpecAugment
        if self.augment and random.random() < 0.5:
            log_mel = spec_augment(log_mel)

        # Force fixed shape
        TARGET_FRAMES = 862  # 10s mel frames approx
        log_mel = log_mel[:, :TARGET_FRAMES]
        if log_mel.shape[1] < TARGET_FRAMES:
            pad = TARGET_FRAMES - log_mel.shape[1]
            log_mel = np.pad(log_mel, ((0,0),(0,pad)))

        log_mel = np.expand_dims(log_mel, 0)
        log_mel = np.repeat(log_mel, 3, axis=0)

        return torch.tensor(log_mel, dtype=torch.float32), torch.tensor(label)

In [ ]:
# DataLoaders
train_dataset = GenreDataset(train_samples, genre_to_label, augment=True)
val_dataset   = GenreDataset(val_samples, genre_to_label, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

In [ ]:
# Model
model = models.efficientnet_b0(pretrained=True)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(genres))

model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
criterion = nn.CrossEntropyLoss()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 138MB/s]


In [ ]:
#training loop
best_acc = 0

for epoch in range(config.epochs):

    model.train()
    train_loss = 0
    train_preds = []
    train_targets = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        outputs = model(x)
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(torch.argmax(outputs,1).cpu().numpy())
        train_targets.extend(y.cpu().numpy())

    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro")

    # Validation
    model.eval()
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            val_preds.extend(torch.argmax(outputs,1).cpu().numpy())
            val_targets.extend(y.cpu().numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    })

    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet.pth")

Epoch 1 | Val Acc: 0.5250
Epoch 2 | Val Acc: 0.6600
Epoch 3 | Val Acc: 0.7450
Epoch 4 | Val Acc: 0.7700
Epoch 5 | Val Acc: 0.7700
Epoch 6 | Val Acc: 0.7900
Epoch 7 | Val Acc: 0.8000
Epoch 8 | Val Acc: 0.8250
Epoch 9 | Val Acc: 0.8250
Epoch 10 | Val Acc: 0.8050
Epoch 11 | Val Acc: 0.8350
Epoch 12 | Val Acc: 0.8200


In [ ]:
model.load_state_dict(torch.load("best_efficientnet.pth"))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
# ===============================
# EfficientNet Robust Inference
# With TTA + W&B Logging
# ===============================

!pip install wandb -q

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import wandb
from torchvision import models

# -------------------------------
# 1️ Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# 2️ W&B Init
# -------------------------------
wandb.login(relogin=True)

wandb.init(
    project="DL-23f100190-notebook-t12026",
    entity="23f1000190-dl-genai-project",
    name="EffNetB0_inference_TTA2",
    config={
        "model": "efficientnet_b0",
        "tta_windows": 5,
        "tta_scales": [1.0, 0.9, 1.1],
        "segment_seconds": 10
    }
)



wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Model loaded


KeyboardInterrupt: 

In [ ]:
# -------------------------------
# 3 Load Model
# -------------------------------
NUM_CLASSES = 10

model = models.efficientnet_b0(pretrained=False)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, NUM_CLASSES)

model.load_state_dict(torch.load("best_efficientnet.pth", map_location=device))
model = model.to(device)
model.eval()

print("Model loaded")

# -------------------------------
# 4️Label Mapping
# -------------------------------
TRAIN_PATH =  "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"  # <-- UPDATE THIS

genres = sorted(os.listdir(TRAIN_PATH))
label_to_genre = {i: g for i, g in enumerate(genres)}

# -------------------------------
# 5️Mel Processing
# -------------------------------
def waveform_to_tensor(waveform):

    mel = librosa.feature.melspectrogram(
        y=waveform,
        sr=44100,
        n_mels=128,
        hop_length=512
    )

    log_mel = librosa.power_to_db(mel)

    TARGET_FRAMES = 862  # 10s
    log_mel = log_mel[:, :TARGET_FRAMES]

    if log_mel.shape[1] < TARGET_FRAMES:
        pad = TARGET_FRAMES - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0,0),(0,pad)))

    log_mel = np.expand_dims(log_mel, 0)
    log_mel = np.repeat(log_mel, 3, axis=0)

    tensor = torch.tensor(log_mel, dtype=torch.float32).unsqueeze(0).to(device)
    return tensor

# -------------------------------
# 6️TTA Prediction
# -------------------------------
def predict_file(model, file_path):

    waveform, _ = librosa.load(file_path, sr=44100)

    SEGMENT = 10 * 44100
    starts = [0, 5*44100, 10*44100, 15*44100, 20*44100]

    logits_sum = 0
    count = 0

    for start in starts:

        if start >= len(waveform):
            continue

        if start + SEGMENT > len(waveform):
            segment = np.pad(
                waveform[start:],
                (0, start + SEGMENT - len(waveform))
            )
        else:
            segment = waveform[start:start+SEGMENT]

        for scale in [1.0, 0.9, 1.1]:

            seg_aug = np.clip(segment * scale, -1, 1)
            tensor = waveform_to_tensor(seg_aug)

            with torch.no_grad():
                logits = model(tensor)

            logits_sum += logits
            count += 1

    logits_avg = logits_sum / count
    pred = torch.argmax(logits_avg, dim=1).item()

    return pred




/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded


In [ ]:
# -------------------------------
# 7️ Generate Submission
# -------------------------------







In [ ]:
TEST_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

In [ ]:
test_df = pd.read_csv(os.path.join(TEST_PATH, "test.csv"))

In [ ]:

predictions = []

for row in test_df.itertuples(index=False):
    file_path = os.path.join(TEST_PATH, row.filename)
    try:
        pred_label = predict_file(model, file_path)
        pred_genre = label_to_genre.get(pred_label, "Unknown")
    except Exception as e:
        pred_genre = f"Error: {e}"
    predictions.append(pred_genre)



KeyboardInterrupt: 

In [ ]:
# Example: attach predictions to dataframe
test_df["genre"] = predictions
test_df.to_csv("submission.csv", index=False)

In [ ]:
predictions = []

for _, row in test_df.iterrows():
    file_path = os.path.join(TEST_PATH, row["filename"])
    pred_label = predict_file(model, file_path)
    pred_genre = label_to_genre[pred_label]
    predictions.append(pred_genre)

KeyboardInterrupt: 

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission_correct.csv", index=False)

print("submission_correct.csv created")

# -------------------------------
# 8️Log to W&B
# -------------------------------
#artifact = wandb.Artifact("efficientnet_submission", type="submission")
#artifact.add_file("submission.csv")
#wandb.log_artifact(artifact)

#wandb.finish()

#print("W&B logging complete")

ValueError: array length 2776 does not match index length 3020

In [ ]:
# Correct fixed mapping (DO NOT CHANGE)
label_to_genre = {
 0: 'rock',
 1: 'reggae',
 2: 'hiphop',
 3: 'pop',
 4: 'jazz',
 5: 'blues',
 6: 'country',
 7: 'classical',
 8: 'disco',
 9: 'metal'
}

TEST_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
test_df = pd.read_csv(os.path.join(TEST_PATH, "test.csv"))

predictions = []

for _, row in test_df.iterrows():
    file_path = os.path.join(TEST_PATH, row["filename"])
    pred_label = predict_file(model, file_path)
    pred_genre = label_to_genre[pred_label]
    predictions.append(pred_genre)

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission_correct.csv", index=False)

print("submission_correct.csv created")

NameError: name 'pd' is not defined

In [ ]:
data_path

'/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

In [10]:
# ==========================================================
# COMPLETE SCRIPT: TRAIN + INFERENCE + SUBMISSION
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import wandb

from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm


# ----------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------

SR = 44100
FULL_LENGTH = 30 * SR
SEGMENT = 10 * SR
N_MELS = 128

BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
TEST_PATH  = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"


# ----------------------------------------------------------
# GENRE MAPPING
# ----------------------------------------------------------

genres = sorted(os.listdir(TRAIN_PATH))

genre_to_label = {g:i for i,g in enumerate(genres)}
label_to_genre = {i:g for g,i in genre_to_label.items()}

print("Genre mapping:", genre_to_label)


# ----------------------------------------------------------
# BUILD TRAIN/VAL LIST
# ----------------------------------------------------------

samples = []
genre_to_folders = {i: [] for i in range(len(genres))}

for genre in genres:

    genre_path = os.path.join(TRAIN_PATH, genre)
    label = genre_to_label[genre]

    for folder in os.listdir(genre_path):

        folder_path = os.path.join(genre_path, folder)

        samples.append((folder_path, label))
        genre_to_folders[label].append(folder_path)

random.shuffle(samples)

split = int(0.8 * len(samples))

train_samples = samples[:split]
val_samples   = samples[split:]

print("Train:", len(train_samples))
print("Val:", len(val_samples))


# ----------------------------------------------------------
# LOAD NOISE FILES
# ----------------------------------------------------------

noise_files = []

for f in os.listdir(NOISE_PATH):
    if f.endswith(".wav"):
        noise_files.append(os.path.join(NOISE_PATH, f))

print("Noise files:", len(noise_files))


# ----------------------------------------------------------
# DATASET
# ----------------------------------------------------------

class GenreDataset(Dataset):

    def __init__(self, samples, genre_to_folders, noise_files=None, augment=False):

        self.samples = samples
        self.genre_to_folders = genre_to_folders
        self.noise_files = noise_files
        self.augment = augment


    def __len__(self):
        return len(self.samples)


    def load_stem(self, folder, stem):

        path = os.path.join(folder, f"{stem}.wav")

        waveform, _ = librosa.load(path, sr=SR)

        if len(waveform) > FULL_LENGTH:
            waveform = waveform[:FULL_LENGTH]
        else:
            waveform = np.pad(waveform, (0, FULL_LENGTH - len(waveform)))

        return waveform


    def __getitem__(self, idx):

        folder_path, label = self.samples[idx]

        if self.augment and random.random() < 0.5:

            folders = self.genre_to_folders[label]

            bass = self.load_stem(random.choice(folders), "bass")
            drums = self.load_stem(random.choice(folders), "drums")
            vocals = self.load_stem(random.choice(folders), "vocals")
            other = self.load_stem(random.choice(folders), "other")

        else:

            bass = self.load_stem(folder_path, "bass")
            drums = self.load_stem(folder_path, "drums")
            vocals = self.load_stem(folder_path, "vocals")
            other = self.load_stem(folder_path, "other")


        waveform = bass + drums + vocals + other
        waveform = np.clip(waveform, -1, 1)


        if self.augment:

            start = random.randint(0, FULL_LENGTH - SEGMENT)
            waveform = waveform[start:start+SEGMENT]

        else:

            waveform = waveform[:SEGMENT]


        if self.augment and self.noise_files:

            noise, _ = librosa.load(random.choice(self.noise_files), sr=SR)

            if len(noise) < SEGMENT:
                noise = np.pad(noise, (0, SEGMENT-len(noise)))

            noise = noise[:SEGMENT]

            alpha = random.uniform(0.02,0.15)

            waveform += alpha*noise


        mel = librosa.feature.melspectrogram(
            y=waveform,
            sr=SR,
            n_mels=N_MELS
        )

        log_mel = librosa.power_to_db(mel)

        log_mel = (log_mel - log_mel.mean())/(log_mel.std()+1e-6)

        log_mel = np.expand_dims(log_mel,0)
        log_mel = np.repeat(log_mel,3,axis=0)

        return torch.tensor(log_mel,dtype=torch.float32), torch.tensor(label)


# ----------------------------------------------------------
# DATALOADERS
# ----------------------------------------------------------

train_dataset = GenreDataset(train_samples, genre_to_folders, noise_files, augment=True)
val_dataset   = GenreDataset(val_samples, genre_to_folders, augment=False)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE)


# ----------------------------------------------------------
# MODEL
# ----------------------------------------------------------

model = models.resnet18(pretrained=True)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features,len(genres))

model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=LR)
criterion = nn.CrossEntropyLoss()


# ----------------------------------------------------------
# WANDB
# ----------------------------------------------------------

wandb.init(project="genre-classification",name="resnet18-final")


# ----------------------------------------------------------
# TRAINING
# ----------------------------------------------------------

best_acc = 0

for epoch in range(EPOCHS):

    model.train()

    train_preds=[]
    train_targets=[]

    for x,y in tqdm(train_loader):

        x=x.to(DEVICE)
        y=y.to(DEVICE)

        optimizer.zero_grad()

        logits=model(x)

        loss=criterion(logits,y)

        loss.backward()

        optimizer.step()

        train_preds+=logits.argmax(1).cpu().tolist()
        train_targets+=y.cpu().tolist()


    train_acc=accuracy_score(train_targets,train_preds)

    model.eval()

    val_preds=[]
    val_targets=[]

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(DEVICE)

            logits=model(x)

            val_preds+=logits.argmax(1).cpu().tolist()
            val_targets+=y.tolist()


    val_acc=accuracy_score(val_targets,val_preds)

    print("Epoch",epoch+1,"Val Acc",val_acc)

    wandb.log({"val_acc":val_acc})


    if val_acc>best_acc:

        best_acc=val_acc

        torch.save(model.state_dict(),"best_model.pth")



# ----------------------------------------------------------
# LOAD BEST MODEL
# ----------------------------------------------------------

model.load_state_dict(torch.load("best_model.pth"))

model.eval()



# ----------------------------------------------------------
# MEL HELPER
# ----------------------------------------------------------

def waveform_to_tensor(waveform):

    mel=librosa.feature.melspectrogram(
        y=waveform,
        sr=SR,
        n_mels=N_MELS
    )

    log_mel=librosa.power_to_db(mel)

    log_mel=(log_mel-log_mel.mean())/(log_mel.std()+1e-6)

    log_mel=np.expand_dims(log_mel,0)
    log_mel=np.repeat(log_mel,3,axis=0)

    log_mel=np.expand_dims(log_mel,0)

    return torch.tensor(log_mel,dtype=torch.float32).to(DEVICE)



# ----------------------------------------------------------
# MULTI CROP INFERENCE
# ----------------------------------------------------------

def predict(file_path):

    waveform,_=librosa.load(file_path,sr=SR)

    if len(waveform)>FULL_LENGTH:
        waveform=waveform[:FULL_LENGTH]
    else:
        waveform=np.pad(waveform,(0,FULL_LENGTH-len(waveform)))

    logits_sum=0

    starts=[0,SEGMENT,2*SEGMENT]

    for start in starts:

        segment=waveform[start:start+SEGMENT]

        tensor=waveform_to_tensor(segment)

        with torch.no_grad():

            logits=model(tensor)

        logits_sum+=logits

    logits_avg=logits_sum/3

    pred=torch.argmax(logits_avg,dim=1).item()

    return pred



# ----------------------------------------------------------
# GENERATE SUBMISSION
# ----------------------------------------------------------

test_df=pd.read_csv(os.path.join(TEST_PATH,"test.csv"))

predictions=[]

for _,row in test_df.iterrows():

    file_path=os.path.join(TEST_PATH,row["filename"])

    pred_label=predict(file_path)

    pred_genre=label_to_genre[pred_label]

    predictions.append(pred_genre)



submission=pd.DataFrame({
    "id":test_df["id"],
    "genre":predictions
})

submission.to_csv("submission.csv",index=False)

print("Submission saved")

Genre mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Train: 800
Val: 200
Noise files: 2000


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


...

Epoch 13 Val Acc 0.87


100%|██████████| 50/50 [11:17<00:00, 13.56s/it]


Epoch 14 Val Acc 0.865


100%|██████████| 50/50 [11:28<00:00, 13.77s/it]


Epoch 15 Val Acc 0.835
Submission saved
